In [2]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
from torch.optim import Adam, LBFGS
import torch.nn.functional as F
import numpy as np
import pygame
import cv2
import flappy_bird_env  # generator module
import argparse
from pygame.locals import QUIT, KEYDOWN, K_ESCAPE, K_SPACE, K_UP


In [ ]:
# === Environment Initialization ===
def init_flappy_env():
    """Initialize pygame via the shared init_pygame() in flappy_bird_env."""
    flappy_bird_env.init_pygame('Flappy Bird (training)')

In [4]:
# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# === Frame Preprocessing ===
def preprocess(frame, out_size=84):
    """
    frame: H×W×3 uint8 numpy array
    returns: torch.FloatTensor [1, out_size, out_size] on device
    """
    import cv2
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (out_size, out_size), interpolation=cv2.INTER_AREA)
    tensor = torch.from_numpy(resized).float().div(255.0).unsqueeze(0).to(device)
    return tensor


# …existing imports…

# replace CNNPolicy with:
class FlappyBirdPolicy(nn.Module):
    def __init__(self, in_ch=1, n_actions=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 16, 8, 4, 2)   # 84->21
        self.conv2 = nn.Conv2d(16, 32, 4, 2, 1)     # 21->10
        self.conv3 = nn.Conv2d(32, 32, 3, 1, 1)     # 10->10
        self.fc1   = nn.Linear(32*10*10, 256)
        self.fc2   = nn.Linear(256, n_actions)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # logits


Using device: cpu


/home/hamlil/anaconda3/envs/flappy/lib/python3.12/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
def compute_returns(rewards, gamma=0.99):
    """
    rewards: 1D torch tensor
    returns: same shape; discounted returns R_t
    """
    T = rewards.size(0)
    pw = gamma ** torch.arange(T, device=rewards.device, dtype=rewards.dtype)
    returns = torch.flip(torch.cumsum(torch.flip(rewards * pw, dims=[0]), dim=0), dims=[0]) / pw
    return returns


In [ ]:
def collect_episode(policy,render=False):
    """
    Returns:
      obs_list: [step tensors]
      act_list: [ints]
      rew_tensor: torch tensor of rewards
      total_reward: float
    """
    init_flappy_env()
    gen = flappy_bird_env.flappygame_generator(action=None)
    frame, _ = next(gen)

    obs_list, act_list, rewards = [], [], []
    done = False
    score = 0

    while not done:
        obs = preprocess(frame)                                 # [1,84,84]
        logits = policy(obs.unsqueeze(0))                       # [1, n_actions]
        dist = Categorical(logits=logits)
        action = dist.sample().item()

        obs_list.append(obs)
        act_list.append(action)

        try:
            frame, new_score = gen.send(action)
        except StopIteration:
            # crash
            rewards.append(-10.0 + (new_score - score))
            done = True
            break

        reward = 0.1 + (new_score - score)                      # alive reward + pipes passed
        rewards.append(reward)
        score = new_score

        if render:
            surf = pygame.surfarray.make_surface(frame.transpose(1, 0, 2))
            flappy_bird_env.window.blit(surf, (0, 0))
            pygame.display.update()
            flappy_bird_env.framepersecond_clock.tick(flappy_bird_env.framepersecond)
            for e in pygame.event.get():
                if e.type == pygame.QUIT:
                    done = True
                    break

    rew_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
    total_reward = float(rew_tensor.sum().item())
    return obs_list, act_list, rew_tensor, total_reward


In [ ]:
def train_policy(policy,
                 optimizer,
                 num_epochs=1000,
                 episodes_per_epoch=10,
                 gamma=0.99,
                 entropy_coef=0.01,
                 grad_clip=10.0,
                 render=False,
                 print_every=1):
    rewards_hist, loss_hist = [], []

    for epoch in range(1, num_epochs+1):
        batch_obs, batch_actions, batch_returns = [], [], []
        batch_rewards = []

        # ---- gather trajectories ----
        for _ in range(episodes_per_epoch):
            obs_list, act_list, rew_t, total_reward = collect_episode(policy, render)
            # compute returns for this episode
            returns = compute_returns(rew_t, gamma)

            batch_obs.extend(obs_list)
            batch_actions.extend(act_list)
            batch_returns.append(returns)
            batch_rewards.append(total_reward)
            rewards_hist.append(total_reward)

        # ---- stack & normalize advantages ----
        obs_tensor     = torch.stack(batch_obs).to(device)            # [N,1,84,84]
        actions_tensor = torch.tensor(batch_actions, dtype=torch.long, device=device)  # [N]
        returns_tensor = torch.cat(batch_returns)                      # [N]

        adv = (returns_tensor - returns_tensor.mean()) / (returns_tensor.std() + 1e-8)

        # ---- forward & loss ----
        logits = policy(obs_tensor)                                    # [N, n_actions]
        dist   = Categorical(logits=logits)
        logp   = dist.log_prob(actions_tensor)                         # [N]
        entropy = dist.entropy().mean()

        policy_loss = -(logp * adv).mean()
        loss = policy_loss - entropy_coef * entropy

        optimizer.zero_grad()
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip)
        optimizer.step()

        loss_val = float(loss.item())
        loss_hist.append(loss_val)

        if epoch % print_every == 0:
            avg_reward = np.mean(batch_rewards)
            print(f"Epoch {epoch}/{num_epochs} - Avg Reward: {avg_reward:.2f}, "
                  f"Loss: {loss_val:.4f}, Entropy: {entropy.item():.3f}")

    return rewards_hist, loss_hist


In [9]:
def evaluate(policy, episodes=10, render=False, deterministic=True):
    policy.eval()
    scores = []
    with torch.no_grad():
        for _ in range(episodes):
            init_flappy_env()
            gen = flappy_bird_env.flappygame_generator(action=None)
            frame, _ = next(gen)
            score = 0
            done = False
            while not done:
                obs = preprocess(frame).unsqueeze(0)  # [1,1,84,84]
                logits = policy(obs)
                if deterministic:
                    action = torch.argmax(logits, dim=1).item()
                else:
                    dist = Categorical(logits=logits)
                    action = dist.sample().item()
                try:
                    frame, new_score = gen.send(action)
                except StopIteration:
                    score = new_score
                    done = True
                    break
                score = new_score

                if render:
                    surf = pygame.surfarray.make_surface(frame.transpose(1, 0, 2))
                    flappy_bird_env.window.blit(surf, (0, 0))
                    pygame.display.update()
                    flappy_bird_env.framepersecond_clock.tick(flappy_bird_env.framepersecond)

    policy.train()
    return scores


In [10]:
def save_checkpoint(policy, optimizer, path="flappy_policy.pt"):
    torch.save({
        'policy_state_dict': policy.state_dict(),
        'optim_state_dict': optimizer.state_dict(),
    }, path)

def load_checkpoint(policy, optimizer, path="flappy_policy.pt", map_location=device):
    data = torch.load(path, map_location=map_location)
    policy.load_state_dict(data['policy_state_dict'])
    optimizer.load_state_dict(data['optim_state_dict'])


In [12]:
if __name__ == "__main__":
    policy = FlappyBirdPolicy(in_ch=1, n_actions=2).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)

    rewards_hist, loss_hist = train_policy(
        policy, optimizer,
        num_epochs=500,
        episodes_per_epoch=10,
        gamma=0.99,
        entropy_coef=0.01,
        grad_clip=10.0,
        render=True,
        print_every=1
    )

    save_checkpoint(policy, optimizer, "flappy_policy.pt")


Epoch 1/500 - Avg Reward: -6.46, Loss: -0.0042, Entropy: 0.692
Epoch 2/500 - Avg Reward: -6.42, Loss: -0.0067, Entropy: 0.693
Epoch 3/500 - Avg Reward: -6.36, Loss: -0.0091, Entropy: 0.693
Epoch 4/500 - Avg Reward: -6.44, Loss: -0.0131, Entropy: 0.690
Epoch 5/500 - Avg Reward: -6.32, Loss: -0.0146, Entropy: 0.684
Epoch 6/500 - Avg Reward: -6.26, Loss: -0.0030, Entropy: 0.674
Epoch 7/500 - Avg Reward: -6.07, Loss: -0.0229, Entropy: 0.662
Epoch 8/500 - Avg Reward: -5.63, Loss: -0.0128, Entropy: 0.644
Epoch 9/500 - Avg Reward: -5.45, Loss: -0.0745, Entropy: 0.619
Epoch 10/500 - Avg Reward: -5.80, Loss: -0.0180, Entropy: 0.587
Epoch 11/500 - Avg Reward: -4.23, Loss: -0.0312, Entropy: 0.548
Epoch 12/500 - Avg Reward: -4.38, Loss: -0.0247, Entropy: 0.498
Epoch 13/500 - Avg Reward: -3.38, Loss: -0.0501, Entropy: 0.439
Epoch 14/500 - Avg Reward: 0.09, Loss: -0.0206, Entropy: 0.370
Epoch 15/500 - Avg Reward: -0.08, Loss: 0.0417, Entropy: 0.297
Epoch 16/500 - Avg Reward: 1.87, Loss: 0.0535, Entr

In [13]:
rewards_hist, loss_hist = train_policy(
        policy, optimizer,
        num_epochs=500,
        episodes_per_epoch=10,
        gamma=0.99,
        entropy_coef=0.01,
        grad_clip=10.0,
        render=True,
        print_every=1
    )

save_checkpoint(policy, optimizer, "flappy_policy.pt")

Epoch 1/500 - Avg Reward: 3.92, Loss: -0.0235, Entropy: 0.322
Epoch 2/500 - Avg Reward: 0.23, Loss: -0.0259, Entropy: 0.327
Epoch 3/500 - Avg Reward: 0.25, Loss: -0.0107, Entropy: 0.327
Epoch 4/500 - Avg Reward: -0.90, Loss: 0.0100, Entropy: 0.327
Epoch 5/500 - Avg Reward: 1.67, Loss: -0.0302, Entropy: 0.331
Epoch 6/500 - Avg Reward: 2.01, Loss: -0.0049, Entropy: 0.329
Epoch 7/500 - Avg Reward: 0.37, Loss: 0.0019, Entropy: 0.327
Epoch 8/500 - Avg Reward: -0.95, Loss: 0.0405, Entropy: 0.328
Epoch 9/500 - Avg Reward: 0.93, Loss: -0.0168, Entropy: 0.338
Epoch 10/500 - Avg Reward: 1.46, Loss: -0.0416, Entropy: 0.346
Epoch 11/500 - Avg Reward: 1.98, Loss: -0.0055, Entropy: 0.345
Epoch 12/500 - Avg Reward: 4.57, Loss: -0.0277, Entropy: 0.345
Epoch 13/500 - Avg Reward: 0.44, Loss: 0.0044, Entropy: 0.340
Epoch 14/500 - Avg Reward: 0.63, Loss: 0.0058, Entropy: 0.338
Epoch 15/500 - Avg Reward: -0.30, Loss: 0.0023, Entropy: 0.338
Epoch 16/500 - Avg Reward: 2.70, Loss: -0.0313, Entropy: 0.342
Epoc

In [17]:
# ——————————————————————————————
# 1) Make sure you’ve already run the cells that:
#    • import torch, pygame, your flappy_bird_env and preprocessing
#    • define FlappyQNN (the CNN-based policy)
#    • define save_checkpoint / load_checkpoint
#    • define evaluate()
#    • set device = torch.device(...)
# ——————————————————————————————

# 2) (Re-)instantiate your policy & optimizer exactly as in training
policy0    = FlappyBirdPolicy(in_ch=1, n_actions=2).to(device)
optimizer = torch.optim.Adam(policy0.parameters(), lr=1e-3)

# 3) Load the weights you saved earlier
load_checkpoint(policy0, optimizer, path="flappy_policy.pt", map_location=device)
print("✅ Checkpoint loaded.")

# 4) Run evaluation: 10 episodes, render to the screen, take the max-logit action each step
scores = evaluate(
    policy0,
    episodes=10,
    render=True,
    deterministic=False
)

# 5) (Optional) Print out the raw scores & average
print("Episode scores:", scores)
print(f"Average score over 10 episodes: {sum(scores)/len(scores):.2f}")


✅ Checkpoint loaded.


KeyboardInterrupt: 